In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 读取您生成的合并数据集
df = pd.read_csv('nhanes_final_merged.csv')

# 2. 定义特征集
# 第一阶段：基础临床特征
stage1_cols = ['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'SMQ020', 'BPQ020']
# 第二阶段：增加实验室指标 SII 和 TyG
stage2_cols = stage1_cols + ['SII', 'TyG']

# 3. 准备数据并进行标准化
X = df[stage2_cols]
y = df['target']

# 采用分层抽样划分 30% 为测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. 训练一阶段模型并计算 AUC
model1 = LogisticRegression(max_iter=1000)
model1.fit(X_train_scaled[:, :len(stage1_cols)], y_train)
auc1 = roc_auc_score(y_test, model1.predict_proba(X_test_scaled[:, :len(stage1_cols)])[:, 1])

# 5. 训练二阶段模型并计算 AUC
model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train_scaled, y_train)
auc2 = roc_auc_score(y_test, model2.predict_proba(X_test_scaled)[:, 1])

# 6. 输出对比结果
print("-" * 40)
print(f"第一阶段 AUC (基础临床模型): {auc1:.4f}")
print(f"第二阶段 AUC (临床 + SII + TyG): {auc2:.4f}")
print(f"AUC 净增量 (ΔAUC): {auc2 - auc1:.4f}")
print("-" * 40)

----------------------------------------
第一阶段 AUC (基础临床模型): 0.8118
第二阶段 AUC (临床 + SII + TyG): 0.8429
AUC 净增量 (ΔAUC): 0.0311
----------------------------------------


In [2]:
import pandas as pd

# 1. 加载现有的合并文件
input_file = 'nhanes_final_merged.csv'
output_file = 'nhanes_clinical_dataset.csv'

df = pd.read_csv(input_file)

# 2. 定义更名映射字典
rename_dict = {
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender',
    'DIQ010': 'Diabetes_Raw',
    'LBXPLTSI': 'Platelets',
    'LBDNENO': 'Neutrophils',
    'LBDLYMNO': 'Lymphocytes',
    'LBXTR': 'Triglycerides',
    'LBXGLU': 'Glucose',
    'BMXBMI': 'BMI',
    'SMQ020': 'Smoking',
    'BPQ020': 'Hypertension',
    'target': 'Diabetes',
    'TyG': 'TyG_Index'
}

# 3. 执行更名
df_renamed = df.rename(columns=rename_dict)

# 4. 重新排列列顺序（让结局变量在最末尾，方便观察）
# 您可以根据需要调整顺序
cols_order = ['SEQN', 'Age', 'Gender', 'BMI', 'Smoking', 'Hypertension', 
              'Platelets', 'Neutrophils', 'Lymphocytes', 'SII', 
              'Triglycerides', 'Glucose', 'TyG_Index', 'Diabetes']

# 只选取存在的列
final_cols = [c for c in cols_order if c in df_renamed.columns]
df_final = df_renamed[final_cols]

# 5. 保存结果
df_final.to_csv(output_file, index=False)

print("-" * 30)
print(f"更名完成！新文件已保存至: {output_file}")
print("前 5 行预览：")
print(df_final.head())

------------------------------
更名完成！新文件已保存至: nhanes_clinical_dataset.csv
前 5 行预览：
       SEQN   Age  Gender   BMI  Smoking  Hypertension  Platelets  \
0  109271.0  49.0     1.0  29.7      1.0           2.0      254.0   
1  109274.0  68.0     1.0  30.2      2.0           1.0      173.0   
2  109282.0  76.0     1.0  26.6      1.0           1.0      154.0   
3  109286.0  33.0     2.0  28.9      2.0           2.0      321.0   
4  109290.0  68.0     2.0  28.1      2.0           1.0      289.0   

   Neutrophils  Lymphocytes          SII  Triglycerides  Glucose  TyG_Index  \
0          3.0          1.8   423.333333           84.0    103.0   8.372399   
1          2.5          0.6   720.833333          133.0    154.0   9.234155   
2          2.6          0.9   444.888889          132.0     95.0   8.743532   
3          7.9          1.9  1334.684211          192.0     92.0   9.086137   
4          4.7          3.5   388.085714          102.0    106.0   8.595265   

   Diabetes  
0       0.0  


In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

# 1. 加载更名后的数据集
df = pd.read_csv('nhanes_clinical_dataset.csv')

# 2. 定义特征集
# 第一阶段 (Base Model)
s1_features = ['Age', 'Gender', 'BMI', 'Smoking', 'Hypertension']
# 第二阶段 (Expanded Model: 增加 SII 和 TyG)
s2_features = s1_features + ['SII', 'TyG_Index']

X = df[s2_features]
y = df['Diabetes']

# 划分训练/测试集 (保持与之前实验一致)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. 训练两个逻辑回归模型
model_old = LogisticRegression(max_iter=1000).fit(X_train_scaled[:, :len(s1_features)], y_train)
model_new = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)

# 获取测试集的预测概率
p_old = model_old.predict_proba(X_test_scaled[:, :len(s1_features)])[:, 1]
p_new = model_new.predict_proba(X_test_scaled)[:, 1]

# ==========================================
# 4. 计算 IDI (综合判别改进指数)
# ==========================================
# IDI = (Slope_new) - (Slope_old) [cite: 407]
# 其中 Slope 为患病组平均概率与非患病组平均概率之差 [cite: 411]
slope_new = np.mean(p_new[y_test == 1]) - np.mean(p_new[y_test == 0])
slope_old = np.mean(p_old[y_test == 1]) - np.mean(p_old[y_test == 0])
idi = slope_new - slope_old

# ==========================================
# 5. 计算 NRI (按照文献推荐的 NRI at event rate)
# ==========================================
# 文献建议将切分阈值 (Threshold) 设为事件率 y [cite: 71, 149, 419]
event_rate = np.mean(y_test)

def get_se_sp(p, y, threshold):
    y_pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    se = tp / (tp + fn)
    sp = tn / (tn + fp)
    return se, sp

se_old, sp_old = get_se_sp(p_old, y_test, event_rate)
se_new, sp_new = get_se_sp(p_new, y_test, event_rate)

# NRI(y) = (Se_new - Se_old) + (Sp_new - Sp_old) [cite: 161, 417]
nri_y = (se_new - se_old) + (sp_new - sp_old)

# ==========================================
# 6. 输出结果
# ==========================================
print("-" * 45)
print(f"当前测试集事件率 (Event Rate): {event_rate:.4f}")
print("-" * 45)
print(f"IDI (Integrated Discrimination Improvement): {idi:.4f}")
print(f"NRI(y) (Net Reclassification Index at event rate): {nri_y:.4f}")
print("-" * 45)

---------------------------------------------
当前测试集事件率 (Event Rate): 0.1688
---------------------------------------------
IDI (Integrated Discrimination Improvement): 0.0892
NRI(y) (Net Reclassification Index at event rate): 0.0349
---------------------------------------------


In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 加载数据集
df = pd.read_csv('nhanes_clinical_dataset.csv')

# 2. 定义特征
s1_features = ['Age', 'Gender', 'BMI', 'Smoking', 'Hypertension']
s2_features = s1_features + ['SII', 'TyG_Index']

X = df[s2_features]
y = df['Diabetes']

# 保持与之前一致的训练/测试集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. 训练模型并获取预测概率
model_old = LogisticRegression(max_iter=1000).fit(X_train_scaled[:, :len(s1_features)], y_train)
model_new = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)

p_old = model_old.predict_proba(X_test_scaled[:, :len(s1_features)])[:, 1]
p_new = model_new.predict_proba(X_test_scaled)[:, 1]

# ==========================================
# 4. 计算连续性 NRI (cNRI)
# ==========================================
# cNRI = NRI_events + NRI_nonevents

# 对于患者组 (Events): 概率上升为正确重分类，概率下降为错误
p_new_events = p_new[y_test == 1]
p_old_events = p_old[y_test == 1]
nri_events = (np.sum(p_new_events > p_old_events) - np.sum(p_new_events < p_old_events)) / len(p_new_events)

# 对于非患者组 (Nonevents): 概率下降为正确重分类，概率上升为错误
p_new_nonevents = p_new[y_test == 0]
p_old_nonevents = p_old[y_test == 0]
nri_nonevents = (np.sum(p_new_nonevents < p_old_nonevents) - np.sum(p_new_nonevents > p_old_nonevents)) / len(p_new_nonevents)

cnri = nri_events + nri_nonevents

# ==========================================
# 5. 输出结果
# ==========================================
print("-" * 45)
print(f"连续性 NRI (continuous NRI): {cnri:.4f}")
print(f"其中患者组改善 (NRI_events): {nri_events:.4f}")
print(f"其中非患者组改善 (NRI_nonevents): {nri_nonevents:.4f}")
print("-" * 45)

---------------------------------------------
连续性 NRI (continuous NRI): 0.5990
其中患者组改善 (NRI_events): 0.1658
其中非患者组改善 (NRI_nonevents): 0.4332
---------------------------------------------


In [5]:
# 查看精确的患病人数和患病率
print(f"总样本量: {len(df_final)}")
print(f"糖尿病患者人数: {df_final['Diabetes'].sum()}")
print(f"实际患病率: {df_final['Diabetes'].mean():.2%}")

总样本量: 3691
糖尿病患者人数: 623.0
实际患病率: 16.88%
